# Independent component analysis (ICA)

_Machine Learning_

**Unmix blended signals back into their separate sources.**

Imagine two people talking and two microphones. Each mic hears a mix of both voices.

     This is the cocktail party problem. We want the separate voices back.

---

This notebook is a practice scaffold for the **Independent component analysis (ICA)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.decomposition import FastICA

rng = np.random.default_rng(0)
t = np.linspace(0, 8, 600)
s1 = np.sin(2 * t)                       # source 1: sine
s2 = np.sign(np.sin(3 * t))             # source 2: square wave
S = np.c_[s1, s2]
S += 0.1 * rng.normal(size=S.shape)

# mix the sources together
A = np.array([[1.0, 0.7], [0.5, 1.0]])
X = S @ A.T

ica = FastICA(n_components=2, random_state=0)
S_hat = ica.fit_transform(X)            # recovered sources

# correlation of each recovered signal with a true source (up to sign/order)
corr = np.corrcoef(S_hat.T, S.T)[:2, 2:]
print("recovered-vs-true correlations:")
print(np.round(corr, 2))

## Visualize the data & results

_From two blended digit-image intensity signals, can ICA recover the original independent sources?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import FastICA

# two real source signals: the average pixel pattern of digits 3 and 8
digits = load_digits()
X, y = digits.data, digits.target
s1 = X[y == 3].mean(0)             # 64-pixel mean image of digit 3
s2 = X[y == 8].mean(0)
S = np.c_[s1, s2]
S = S - S.mean(0)

# mix the sources together, then unmix with ICA
A = np.array([[1.0, 0.7], [0.5, 1.0]])
Xmix = S @ A.T
S_hat = FastICA(n_components=2, random_state=0, max_iter=2000).fit_transform(Xmix)

def scale(v):
    m = np.abs(v).max()
    return v / m if m > 0 else v

t = np.arange(64)
plt.plot(t, scale(Xmix[:, 0]), color="#ff7b72", label="mixture")
plt.plot(t, scale(S_hat[:, 0]), color="#4ea1ff", label="recovered source 1")
plt.plot(t, scale(S_hat[:, 1]), color="#7ee787", label="recovered source 2")
plt.xlabel("pixel index")
plt.ylabel("intensity (scaled)")
plt.title("ICA unmixing of digit-pixel signals")
plt.legend()
plt.show()